# Valar LLM — Phase 0 free online smoke

Synthetic examples only; this is not a quality benchmark. Choose a **free GPU runtime** manually.
No paid resources, teacher API calls, automatic Hub upload, or public serving from Colab.
Run `python scripts/prepare_bundle.py notebook` on the development machine and upload its source zip below.
The notebook installs its own Python 3.12.12 environment, independently of Colab's notebook interpreter.
If restoring an already-downloaded adapter in a fresh session, run only source upload and environment setup, then jump to **Fresh-session restore**. Do not repeat baseline inference or training.


In [ ]:
import io
import pathlib
import sys
import zipfile

from google.colab import files

print('Colab notebook Python:', sys.version.split()[0])
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only the prepared source ZIP.'
source_name, source_data = next(iter(uploaded.items()))
assert source_name.startswith('valar-phase0-source') and source_name.endswith('.zip')
root = pathlib.Path('/content/valar-phase0')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_data)) as archive:
    for member in archive.infolist():
        path = (root / member.filename).resolve()
        assert path.is_relative_to(root.resolve()), 'Unsafe archive member'
    archive.extractall(root)
%cd /content/valar-phase0


In [ ]:
import os
import subprocess

# Keep bootstrap tooling separate from Colab's preinstalled Python packages.
bootstrap = '/content/valar-bootstrap'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--target', bootstrap,
                'uv==0.11.7'], check=True)
runner_env = dict(os.environ, PYTHONPATH=bootstrap)
uv = [sys.executable, '-m', 'uv']
subprocess.run(uv + ['python', 'install', '3.12.12'], env=runner_env, check=True)
subprocess.run(uv + ['venv', '--python', '3.12.12', '/content/valar-env'],
               env=runner_env, check=True)
python = '/content/valar-env/bin/python'
subprocess.run(uv + ['pip', 'install', '--python', python, '-r', 'requirements/gpu.lock'],
               env=runner_env, check=True)
subprocess.run(uv + ['pip', 'install', '--python', python, '--no-deps', '-e', '.'],
               env=runner_env, check=True)
subprocess.run([python, '-c', 'import sys, torch; print("Valar Python:", sys.version); assert torch.cuda.is_available(); print(torch.cuda.get_device_name())'], check=True)
os.environ['VALAR_FREE_ONLINE_RUNTIME'] = 'confirmed'
subprocess.run([python, '-m', 'valar_llm.cli', 'check'], check=True)


## Untuned baseline smoke
This makes three bounded real inference attempts (research, review, image answer). Read the saved predictions; valid JSON alone is not correctness. Stop and preserve failures if the model does not fit.
Each retry uses a new timestamped result file. Debug tracebacks are enabled only for these synthetic fixtures.


In [ ]:
from datetime import datetime, timezone

baseline_output = 'results/runs/base-smoke-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json'
result = subprocess.run([python, '-m', 'valar_llm.eval.smoke', '--output', baseline_output,
                         '--debug-errors'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout[-20000:])
if pathlib.Path(baseline_output).exists():
    print(pathlib.Path(baseline_output).read_text())
result.check_returncode()


## Tiny image training smoke
Three steps on one synthetic image. The command verifies exact prompt-prefix masking, image tensors and nonempty answer loss. Training and re-evaluating this example measures mechanics only.


In [ ]:
subprocess.run([python, '-m', 'valar_llm.train.smoke'], check=True)
print(pathlib.Path('artifacts/smoke-adapter/mask-audit.json').read_text())


In [ ]:
import shutil

shutil.make_archive('/content/valar-smoke-adapter', 'zip', 'artifacts', 'smoke-adapter')
files.download('/content/valar-smoke-adapter.zip')
shutil.make_archive('/content/valar-smoke-results', 'zip', 'results/runs')
files.download('/content/valar-smoke-results.zip')


## Fresh-session restore
After downloading both archives, restart the runtime/session. Re-run only the source upload and environment setup cells. Skip base/training cells. Upload the saved adapter archive below, then run reload inference. This checks restore into a new process; record that the VM/session was actually restarted.
Paste the adapter archive's SHA-256 when prompted. This cell audits the saved tensors before extraction, then runs only the synthetic image question through the restored adapter. Its result checks restoration mechanics, not unseen-data improvement.


In [ ]:
import hashlib
from datetime import datetime, timezone

uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only the saved adapter ZIP.'
adapter_data = next(iter(uploaded.values()))
expected_sha = input('Adapter ZIP SHA-256: ').strip().lower()
assert len(expected_sha) == 64 and hashlib.sha256(adapter_data).hexdigest() == expected_sha, 'Checksum mismatch'
restore_zip = pathlib.Path('/content/valar-restored-adapter.zip')
restore_zip.write_bytes(adapter_data)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
audit_output = 'results/runs/restored-checkpoint-audit-' + stamp + '.json'
audit = subprocess.run([python, '-m', 'valar_llm.eval.checkpoint', str(restore_zip),
                        '--expected-sha256', expected_sha, '--output', audit_output],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(audit.stdout)
audit.check_returncode()
destination = pathlib.Path('artifacts').resolve()
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(adapter_data)) as archive:
    for member in archive.infolist():
        assert (destination / member.filename).resolve().is_relative_to(destination)
    archive.extractall(destination)
reload_output = 'results/runs/reloaded-image-smoke-' + stamp + '.json'
restored = subprocess.run([python, '-m', 'valar_llm.eval.smoke', '--adapter', 'artifacts/smoke-adapter',
                           '--case', 'vision-image', '--output', reload_output, '--debug-errors'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(restored.stdout[-20000:])
if pathlib.Path(reload_output).exists():
    print(pathlib.Path(reload_output).read_text())
restored.check_returncode()


## Cloud serving gate
Colab is not the inference server. Follow `docs/account-setup.md` to prepare an eligible free ZeroGPU Space and restore the adapter there. A new Hugging Face account may have to wait for eligibility. Keep this gate unpassed until remote text/image requests actually work, including a probe from a cloud client while the laptop is off.

To run the remote probe here later, install `requirements/probe.lock` into the separate environment, load secrets from Colab Secrets into environment variables without printing them, and run `python scripts/remote_probe.py owner/space-name`. Save quota before/after and cold/warm observations separately.
